## Define libraries

In [1]:
import sys
import os

# Get the absolute path to the parent directory
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Add the parent directory to sys.path if it's not already there
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

In [2]:
from datasets import load_dataset
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
# from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

from helper import RAGHelper
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from dotenv import load_dotenv
load_dotenv()
import os
from datasets import load_dataset

import sqlite3
import pandas as pd
import re
import json

import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.metrics import f1_score

In [ ]:
# ingestion
# embedding_model - 2
# chunking_size - 256, 512, 1024
# chunking_overlap - 200, 100, 50
# search_type - 2
# search_k - 3, 5, 7
# vector_database - 3
# retrieval techniques - 2
# reranking
# gen_model - 2



## Define Variables

In [79]:
dataset_source = 'rungalileo/ragbench'
dataset_name = 'finqa'
data_split = 'test'

chunking_size = 1024
chunking_overlap = 200
separators = ["\n\n", "\n", " ", ".", ","]
# embedding
vector_database = 'chroma' 
chromadb_folder = "../database"
db_name = f"finance_{chunking_size}_{chunking_overlap}"
persist_directory = f"{chromadb_folder}/{db_name}"
embedding_model = "BAAI/bge-base-en-v1.5"

API_KEY = "GROQ_API_KEY"

search_type = "similarity"
search_kwargs = {"k":3}

gen_model = "llama-3.1-8b-instant"

eval_model_type = "groq"
eval_model = "meta-llama/llama-4-scout-17b-16e-instruct"
eval_sample = 40

sqlite_db = "../sqldb/ragproject.db"
table_name = "ragproject_table"



## Data Fetching

In [4]:

dataset = load_dataset(dataset_source, dataset_name, split=data_split)

In [5]:
def deduplicate_data(data):
    data_dict = {}
    for d in data:
        # print(d)
        document = " ".join(d["documents"])
        if document in data_dict:
            data_dict[document]["docid"].append(d["id"])
        else:
            # print(d)
            # break
            data_dict[document] = {"docid":[d["id"]]}
    return data_dict

In [6]:
dedup = deduplicate_data(dataset)

In [7]:
docs = [
    Document(
        
            metadata=v, 
            page_content=k
        
        
    )
    for k,v in dedup.items()
]

## Data Ingestion

In [73]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunking_size, chunk_overlap=chunking_overlap, separators=separators)
docs_chunks = text_splitter.split_documents(docs)

In [74]:
if os.path.exists(persist_directory) and os.listdir(persist_directory):
    print("Loading existing vector database...")
    vector_db = Chroma(
        persist_directory=persist_directory, 
        embedding_function=HuggingFaceEmbeddings(model_name=embedding_model)
    )
else:
    print("Creating new vector database...")
    # This assumes you have 'docs_chunks' already defined as per your notebook
    vector_db = Chroma.from_documents(
        documents=docs_chunks, 
        embedding=HuggingFaceEmbeddings(model_name=embedding_model), 
        persist_directory=persist_directory
    )

Loading existing vector database...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [17]:
# vector_db = Chroma.from_documents(documents=docs_chunks, 
#                                   embedding=HuggingFaceEmbeddings(model_name=embedding_model),
#                                   persist_directory=persist_directory)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## Data Retrieval & Inference

In [75]:
api_key = os.getenv(API_KEY)
rg = RAGHelper(api_key=api_key)


In [76]:
retriever = vector_db.as_retriever(search_type=search_type, search_kwargs=search_kwargs)

In [77]:
eval_message = rg.eval_message

In [78]:
import time

start = time.time()
rg.db_insert(dataset=dataset, eval_message=eval_message, retriever=retriever,
          eval_model_type=eval_model_type, eval_model=eval_model, sample=eval_sample, 
            sqldb=sqlite_db, chunk_size=chunking_size,
            dataset_name=dataset_name, vector_db=vector_database,
             embed_model=embedding_model, chunk_overlap=chunking_overlap,
             gen_model=gen_model, table_name=table_name
             
            )

print("Time:", time.time() - start)

1/40 completed
2/40 completed
3/40 completed
4/40 completed
5/40 completed
Error code: 429 - {'error': {'message': 'Rate limit reached for model `meta-llama/llama-4-scout-17b-16e-instruct` in organization `org_01kt8k53n5ey6s7ehnh37cnf5k` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 498558, Requested 2622. Please try again in 3m23.904s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Time: 85.06899285316467


## Calculating Metrics

In [90]:
conn = sqlite3.connect(sqlite_db)
cursor = conn.cursor()

In [91]:
df = pd.read_sql_query("select * from ragproject_table", conn)
# Close the existing connection
if 'conn' in globals():
    conn.close()
    print("Connection closed. The database should now be unlocked.")

In [92]:
def parse_custom_metadata(text):
    # Remove braces and newlines
    text = text.replace('{', '').replace('}', '').replace('\n', '')
    # Split by comma
    items = text.split(',')
    meta_dict = {}
    for item in items:
        if ':' in item:
            key, value = item.split(':', 1)
            meta_dict[key.strip()] = value.strip()
    return meta_dict


In [93]:
# 1. Apply the custom parser
df['metadata_dict'] = df['metadata'].apply(parse_custom_metadata)

# 2. Normalize and concat
metadata_df = pd.json_normalize(df['metadata_dict'])
df_final = pd.concat([df.drop(columns=['metadata', 'metadata_dict']), metadata_df], axis=1)

### filter data from database

In [94]:
filter_conditions = ((df_final['eval_model']==eval_model) 
                     & 
                     (df_final['chunk_size']=='1024')
                     &(df_final['retriever_search_type']=='similarity')
                    )

In [96]:
df_final['chunk_size'].unique()

<ArrowStringArray>
['1024', '512']
Length: 2, dtype: str

In [51]:
str(chunking_size)

'1024'

In [52]:
df_final.tail()

,id,question,gpt_model_adherence,gpt_model_relevance_score,gpt_model_utilization_score,gpt_model_completeness_score,claude_model_adherence,claude_model_relevance_score,claude_model_utilization_score,claude_model_completeness_score,...,completeness_score,chunk_size,dataset_name,vector_db,embed_model,chunk_overlap,gen_model,eval_model,retriever_search_type,retriever_search_args
147,221,"in billions , what was the total for 2015 and ...",1,0.058824,0.029412,0.5,1,0.029412,0.029412,1.0,...,1.000000,1024,finqa,chroma,BAAI/bge-base-en-v1.5,200,llama-3.1-8b-instant,meta-llama/llama-4-scout-17b-16e-instruct,similarity,'k': 3
148,222,what was the percentage change in net earnings...,1,0.040000,0.040000,1.0,1,0.040000,0.040000,1.0,...,0.000000,1024,finqa,chroma,BAAI/bge-base-en-v1.5,200,llama-3.1-8b-instant,meta-llama/llama-4-scout-17b-16e-instruct,similarity,'k': 3
149,223,how many total shares were repurchase in the p...,1,0.040000,0.040000,1.0,1,0.040000,0.040000,1.0,...,0.000000,1024,finqa,chroma,BAAI/bge-base-en-v1.5,200,llama-3.1-8b-instant,meta-llama/llama-4-scout-17b-16e-instruct,similarity,'k': 3
150,224,without employee severance costs in 2004 and 2...,1,0.085714,0.085714,1.0,1,0.057143,0.057143,1.0,...,0.333333,1024,finqa,chroma,BAAI/bge-base-en-v1.5,200,llama-3.1-8b-instant,meta-llama/llama-4-scout-17b-16e-instruct,similarity,'k': 3
151,225,what is the money pool activity use of operati...,1,0.100000,0.100000,1.0,1,0.100000,0.100000,1.0,...,0.500000,1024,finqa,chroma,BAAI/bge-base-en-v1.5,200,llama-3.1-8b-instant,meta-llama/llama-4-scout-17b-16e-instruct,similarity,'k': 3


In [65]:
df_model = df_final[filter_conditions]

In [66]:
df_model.shape

(52, 23)

In [55]:
df_model.head()

,id,question,gpt_model_adherence,gpt_model_relevance_score,gpt_model_utilization_score,gpt_model_completeness_score,claude_model_adherence,claude_model_relevance_score,claude_model_utilization_score,claude_model_completeness_score,...,completeness_score,chunk_size,dataset_name,vector_db,embed_model,chunk_overlap,gen_model,eval_model,retriever_search_type,retriever_search_args
30,104,what is the rate of return in cadence design s...,1,0.111111,0.111111,1.0,1,0.222222,0.111111,0.5,...,0.250000,1024,finqa,chroma,BAAI/bge-base-en-v1.5,200,llama-3.1-8b-instant,meta-llama/llama-4-scout-17b-16e-instruct,similarity,'k': 3
31,105,what is the ratio of the total american person...,1,0.040000,0.040000,1.0,1,0.040000,0.040000,1.0,...,0.142857,1024,finqa,chroma,BAAI/bge-base-en-v1.5,200,llama-3.1-8b-instant,meta-llama/llama-4-scout-17b-16e-instruct,similarity,'k': 3
32,106,what portion of total assets acquired of anios...,1,0.100000,0.050000,0.5,1,0.100000,0.050000,0.5,...,0.750000,1024,finqa,chroma,BAAI/bge-base-en-v1.5,200,llama-3.1-8b-instant,meta-llama/llama-4-scout-17b-16e-instruct,similarity,'k': 3
33,107,what is the five year total return on the gold...,0,0.111111,0.111111,1.0,1,0.222222,0.111111,0.5,...,1.000000,1024,finqa,chroma,BAAI/bge-base-en-v1.5,200,llama-3.1-8b-instant,meta-llama/llama-4-scout-17b-16e-instruct,similarity,'k': 3
34,108,what is the percentage change in the total car...,1,0.050000,0.050000,1.0,1,0.050000,0.050000,1.0,...,0.666667,1024,finqa,chroma,BAAI/bge-base-en-v1.5,200,llama-3.1-8b-instant,meta-llama/llama-4-scout-17b-16e-instruct,similarity,'k': 3


In [56]:
len(df_model)

70

In [28]:
df_model['relevance_gain_gpt'] = df_model['relevance_score'] - df_model['gpt_model_relevance_score']
df_model['relevance_gain_claude'] = df_model['relevance_score'] - df_model['claude_model_relevance_score']

In [29]:

def calculate_metrics(df_llama):

    # Replace 'column1' and 'column2' with your actual column names
    rmse_relevance_gpt = np.sqrt(mean_squared_error(df_llama['gpt_model_relevance_score'], df_llama['relevance_score']))
    
    print(f"RMSE_relevance_gpt: {rmse_relevance_gpt}")
    
    rmse_relevance_claude = np.sqrt(mean_squared_error(df_llama['claude_model_relevance_score'], df_llama['relevance_score']))
    
    print(f"RMSE_relevance_claude: {rmse_relevance_claude}")
    rmse_utilization_gpt = np.sqrt(mean_squared_error(df_llama['gpt_model_utilization_score'], df_llama['utilization_score']))
    
    print(f"RMSE_utilization_gpt: {rmse_utilization_gpt}")
    
    rmse_utilization_claude = np.sqrt(mean_squared_error(df_llama['claude_model_utilization_score'], df_llama['utilization_score']))
    
    print(f"RMSE_utilization_claude: {rmse_utilization_claude}")
    rmse_completeness_gpt = np.sqrt(mean_squared_error(df_llama['gpt_model_completeness_score'], df_llama['completeness_score']))
    
    print(f"RMSE_completeness_gpt: {rmse_completeness_gpt}")
    
    rmse_completeness_claude = np.sqrt(mean_squared_error(df_llama['claude_model_completeness_score'], df_llama['completeness_score']))
    
    print(f"RMSE_completeness_claude: {rmse_completeness_claude}")

    f1score_gpt = f1_score(df_llama['gpt_model_adherence'].astype(int), df_llama['adherence'].astype(int))

    print(f"F1_adherence_gpt: {f1score_gpt}")

    f1score_claude = f1_score(df_llama['claude_model_adherence'].astype(int), df_llama['adherence'].astype(int))

    print(f"F1_adherence_claude: {f1score_claude}")
    

calculate_metrics(df_model)

RMSE_relevance_gpt: 0.3454732291084483
RMSE_relevance_claude: 0.32383784708074836
RMSE_utilization_gpt: 0.1547670514907414
RMSE_utilization_claude: 0.15280654145959924
RMSE_completeness_gpt: 0.4480044864833201
RMSE_completeness_claude: 0.4753378957618239
F1_adherence_gpt: 0.6285714285714286
F1_adherence_claude: 0.6216216216216216


In [30]:
print(f"relevance_gain_gpt: {df_model['relevance_gain_gpt'].mean()}")
print(f"relevance_gain_claude: {df_model['relevance_gain_claude'].mean()}")

relevance_gain_gpt: 0.2056048552180476
relevance_gain_claude: 0.22383524336512098
